In [1]:
!git clone https://github.com/Brayan695/RNAseq-AMD.git

Cloning into 'RNAseq-AMD'...
remote: Enumerating objects: 2627, done.
remote: Counting objects: 100% (647/647), done.
remote: Compressing objects: 100% (474/474), done.
remote: Total 2627 (delta 282), reused 511 (delta 166), pack-reused 1980 (from 1)
Receiving objects: 100% (2627/2627), 1.04 GiB | 29.32 MiB/s, done.
Resolving deltas: 100% (941/941), done.
Updating files: 100% (1834/1834), done.


In [2]:
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import f_classif
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [3]:
DATA_PATH = "/kaggle/working/RNAseq-AMD/Dataset/MetaSheet_Processed.csv"
N_ITERATIONS = 1000
SAMPLE_FRACTION = 0.8
TOP_K_FRACTION = 0.30     #per iteration: keep top 30% of usable features by score
TOP_N_FRACTION = 0.10     #final: keep top 10% of usable features by selection frequency                                      
RANDOM_SEED = 2026

# All 6 pairwise MGS stage comparisons to run. Each tuple is (negative_class_stage, positive_class_stage) 
# for ex: (1, 4) means "control (MGS1) vs. late AMD (MGS4)"
STAGE_PAIRS = [(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]

EXCLUDE_LABEL_LEAKAGE = True
LEAKAGE_COLUMNS = [
    "oc_AMD",
    "oc_dry AMD",
    "oc_macular degeneration",
    "oc_Wet AMD",
    "oc_wet AMD",
    "oc_early AMD",
    "oc_possible AMD",
    "oc_possible macular degeneration",
    "oc_AMD (received shots)",
]

EXCLUDE_CONFOUNDS = True
CONFOUND_COLUMNS = ["age"]

rng = np.random.default_rng(RANDOM_SEED)

In [4]:
def load_data(path, stage_pair=(1, 4)):
    """
    Load metadata, filter to two MGS stages, split into features (X) and
    label (y). y = 1 for the higher numbered (more advanced) stage in
    stage_pair, 0 for the lower numbered stage. All other stages are
    excluded from this comparison entirely.
    """
    neg_stage, pos_stage = stage_pair
    df = pd.read_csv(path)
    df = df[df["mgs_level"].isin([neg_stage, pos_stage])].reset_index(drop=True)
    y = (df["mgs_level"] == pos_stage).astype(int).values
    drop_cols = [c for c in ["sample_id", "mgs_level"] if c in df.columns]
    X = df.drop(columns=drop_cols)
    return X, y

In [5]:
def drop_constant_features(X):
    """Remove features with zero variance"""
    nunique = X.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    X_filtered = X.drop(columns=constant_cols)
    return X_filtered, constant_cols

In [6]:
def anova_scores(X, y):
    """ANOVA F-test score per feature, all features at once."""
    f_stat, _ = f_classif(X.values, y)
    f_stat = np.nan_to_num(f_stat, nan=0.0)
    return pd.Series(f_stat, index=X.columns)
 
def auc_scores(X, y):
    """AUC per feature, computed directly from ranks (equivalent to
    roc_auc_score but vectorized across every column at once)
    Take max(auc, 1-auc) so the direction of the association (e.g.
    presence vs. absence of a condition) doesn't penalize the score."""
    n1 = y.sum()
    n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return pd.Series(0.5, index=X.columns)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    sum_ranks_pos = ranks[y == 1].sum(axis=0)
    auc = (sum_ranks_pos - n1 * (n1 + 1) / 2) / (n1 * n0)
    auc = np.maximum(auc, 1 - auc)
    return pd.Series(auc, index=X.columns)
 
def kruskal_scores(X, y):
    """Kruskal Wallis H statistic per feature  computed directly from ranks"""
    N = len(y)
    ranks = stats.rankdata(X.values, axis=0, method="average")
    n1 = y.sum()
    n0 = N - n1
    R1 = ranks[y == 1].sum(axis=0)
    R0 = ranks[y == 0].sum(axis=0)
    H = (12 / (N * (N + 1))) * ((R1 ** 2) / n1 + (R0 ** 2) / n0) - 3 * (N + 1)
    # Tie correction: C = 1 - sum(t^3 - t) / (N^3 - N) computed per column
    tie_correction = np.ones(X.shape[1])
    X_vals = X.values
    for i in range(X.shape[1]):
        _, counts = np.unique(X_vals[:, i], return_counts=True)
        tie_sum = np.sum(counts ** 3 - counts)
        tie_correction[i] = 1 - tie_sum / (N ** 3 - N)
    with np.errstate(divide="ignore", invalid="ignore"):
        H_corrected = np.where(tie_correction > 0, H / tie_correction, 0.0)
    H_corrected = np.nan_to_num(H_corrected, nan=0.0, posinf=0.0, neginf=0.0)
    return pd.Series(H_corrected, index=X.columns)

In [7]:
def run_pipeline(X, y, n_iterations, sample_fraction, top_k, seed):
    """Run the 1000 iteration resampling + scoring loop.
    Returns a DataFrame of selection frequency (0-1) per feature per method.
    """
    counts = {
        "anova": pd.Series(0, index=X.columns, dtype=int),
        "auc": pd.Series(0, index=X.columns, dtype=int),
        "kruskal": pd.Series(0, index=X.columns, dtype=int),
    }
    for it in range(n_iterations):
        #Stratified 80% resample: preserves the control to AMD ratio in
        # every iteration so small class features aren't starved.
        X_sub, _, y_sub, _ = train_test_split(
            X, y,
            train_size=sample_fraction,
            stratify=y,
            random_state=RANDOM_SEED + it,
        )
        scores = {
            "anova": anova_scores(X_sub, y_sub),
            "auc": auc_scores(X_sub, y_sub),
            "kruskal": kruskal_scores(X_sub, y_sub),
        }
        for method, s in scores.items():
            top_features = s.sort_values(ascending=False).head(top_k).index
            counts[method].loc[top_features] += 1
        if (it + 1) % 100 == 0:
            print(f"  iteration {it + 1}/{n_iterations} done")
    freq = pd.DataFrame({m: c / n_iterations for m, c in counts.items()})
    return freq

In [8]:
def select_consistent_features(freq_df, top_n):
    """Take the top N features per method (by selection frequency), then
    intersect across all three methods."""
    top_sets = {}
    for method in freq_df.columns:
        top_sets[method] = set(
            freq_df[method].sort_values(ascending=False).head(top_n).index
        )
    consistent = top_sets["anova"] & top_sets["auc"] & top_sets["kruskal"]
    return consistent, top_sets

In [9]:
all_results = {}
 
for stage_pair in STAGE_PAIRS:
    neg_stage, pos_stage = stage_pair
    label = f"{neg_stage}v{pos_stage}"
    print(f"\n{'='*60}\nStage comparison: MGS{neg_stage} vs. MGS{pos_stage}\n{'='*60}")
 
    X_raw, y = load_data(DATA_PATH, stage_pair=stage_pair)
    print(f"  {X_raw.shape[0]} samples, {X_raw.shape[1]} raw features")
    print(f"  class balance: {sum(y==0)} MGS{neg_stage}, {sum(y==1)} MGS{pos_stage}")
 
    if EXCLUDE_LABEL_LEAKAGE:
        present = [c for c in LEAKAGE_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} label-leakage columns (chart notes "
              f"that restate the AMD diagnosis): {present}")
 
    if EXCLUDE_CONFOUNDS:
        present = [c for c in CONFOUND_COLUMNS if c in X_raw.columns]
        X_raw = X_raw.drop(columns=present)
        print(f"  dropped {len(present)} confound columns: {present}")
 
    X, dropped = drop_constant_features(X_raw)
    print(f"  dropped {len(dropped)} constant (zero-variance) features")
    print(f"  {X.shape[1]} usable features remain")
 
    top_k = max(1, int(X.shape[1] * TOP_K_FRACTION))
    top_n = max(1, int(X.shape[1] * TOP_N_FRACTION))
    print(f"  per-iteration top-K = {top_k}, final top-N per method = {top_n}")
 
    freq_df = run_pipeline(X, y, N_ITERATIONS, SAMPLE_FRACTION, top_k, RANDOM_SEED)
    consistent, top_sets = select_consistent_features(freq_df, top_n)
 
    print(f"\n  Features selected by all 3 methods: {len(consistent)}")
    for f in sorted(consistent):
        print(f"    {f}  (anova={freq_df.loc[f,'anova']:.2f}, "
              f"auc={freq_df.loc[f,'auc']:.2f}, kruskal={freq_df.loc[f,'kruskal']:.2f})")
 
    freq_df.to_csv(f"metadata_feature_selection_frequencies_{label}.csv")
    pd.Series(sorted(consistent), name="feature").to_csv(
        f"metadata_selected_features_{label}.csv", index=False
    )
    print(f"  Saved: metadata_feature_selection_frequencies_{label}.csv, "
          f"metadata_selected_features_{label}.csv")
 
    all_results[label] = consistent


Stage comparison: MGS1 vs. MGS2
  280 samples, 614 raw features
  class balance: 105 MGS1, 175 MGS2
  dropped 9 label-leakage columns (chart notes that restate the AMD diagnosis): ['oc_AMD', 'oc_dry AMD', 'oc_macular degeneration', 'oc_Wet AMD', 'oc_wet AMD', 'oc_early AMD', 'oc_possible AMD', 'oc_possible macular degeneration', 'oc_AMD (received shots)']
  dropped 1 confound columns: ['age']
  dropped 175 constant (zero-variance) features
  429 usable features remain
  per-iteration top-K = 128, final top-N per method = 42
  iteration 100/1000 done
  iteration 200/1000 done
  iteration 300/1000 done
  iteration 400/1000 done
  iteration 500/1000 done
  iteration 600/1000 done
  iteration 700/1000 done
  iteration 800/1000 done
  iteration 900/1000 done
  iteration 1000/1000 done

  Features selected by all 3 methods: 16
    mh_-  (anova=0.99, auc=0.90, kruskal=0.99)
    mh_CML  (anova=0.81, auc=0.80, kruskal=0.81)
    mh_ESLD  (anova=0.82, auc=0.81, kruskal=0.82)
    mh_asthma  (anov

In [10]:

#which features are robust across MULTIPLE stage comparisons not just one?

all_features = sorted(set().union(*all_results.values()))
summary = pd.DataFrame(index=all_features)
for label, feats in all_results.items():
    summary[label] = summary.index.isin(feats)
summary["n_pairs_selected"] = summary.sum(axis=1)
summary = summary.sort_values("n_pairs_selected", ascending=False)
 
print(f"\n{'='*60}\nCross-pair summary\n{'='*60}")
print(summary)
summary.to_csv("metadata_feature_selection_cross_pair_summary.csv")
print("\nSaved: metadata_feature_selection_cross_pair_summary.csv")


Cross-pair summary
                                  1v2    1v3    1v4    2v3    2v4    3v4  \
oc_cataracts (OU)                True   True   True   True   True  False   
Y402H_CC                        False   True   True   True   True  False   
A69S_GG                         False  False   True   True   True   True   
mh_high chol                     True   True   True  False   True  False   
oc_confirmed pseudophakic (OU)  False   True   True   True   True  False   
...                               ...    ...    ...    ...    ...    ...   
oc_confirmed pseudophakic (OD)  False   True  False  False  False  False   
oc_on Ocuvite                   False  False  False  False  False   True   
oc_phakic                       False   True  False  False  False  False   
oc_retinal detachment           False  False  False  False  False   True   
oc_retinal injection            False  False  False  False  False   True   

                                n_pairs_selected  
oc_cataracts (OU